# NB08 — Product Analytics Interview Practice

Practice the 5-phase product analytics lifecycle with Claude-powered coaching.
Write **or speak** your response at each phase, then get rubric-based feedback.

In [1]:
import os, sys, uuid
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── Module imports ──
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
for mod in ['interview_practice_utils', 'ui_components']:
    if mod in sys.modules:
        del sys.modules[mod]

import interview_practice_utils as ipu
import ui_components as ui

# ── Paths ──
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
OUTPUTS_DIR  = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'nb08')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# ── Inject custom CSS + render header ──
ui.inject_styles()
ui.render_header()

print(f"  {len(ipu.PHASES)} phases | {len(ipu.COMPANY_ARCHETYPES) * len(ipu.PRODUCT_SITUATIONS) * len(ipu.CONSTRAINT_TWISTS)} scenario combinations")

  5 phases | 480 scenario combinations


In [2]:
# ── Initialize Claude + session ──
claude_ready = ipu.init_claude()
ipu.init_session_logger(OUTPUTS_DIR)
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"

if not claude_ready:
    display(HTML('''<div style="background:#fef2f2; border-left:4px solid #dc2626; padding:12px; border-radius:6px;">
        <strong>Claude not available.</strong> Set ANTHROPIC_API_KEY and re-run this cell.</div>'''))
else:
    display(HTML(f'''<div style="background:#dcfce7; border-left:4px solid #16a34a; padding:10px; border-radius:6px; font-size:13px;">
        Claude ready | Session: <code>{SESSION_ID}</code></div>'''))

Claude ready (claude-sonnet-4-20250514)


In [3]:
# ── Scenario Generator ──
display(HTML('<div class="ipu-section"><h2>Generate a Scenario</h2><p>Choose focus and mode, then generate a random case study.</p></div>'))

_state = {'scenario': None, 'phase_results': {}}

scope_selector = widgets.RadioButtons(
    options=[('Any (random)', None), ('New Product Launch', 'product'), ('Feature on Existing Product', 'feature')],
    value=None, description='Focus:', style={'description_width': '60px'}, layout=widgets.Layout(width='380px'))

mode_selector = widgets.RadioButtons(
    options=[('General Tech', 'general'), ('SmarterDx Prep (Healthcare AI)', 'smarterdx')],
    value='general', description='Mode:', style={'description_width': '60px'}, layout=widgets.Layout(width='380px'))

generate_btn = widgets.Button(description='Generate Scenario', button_style='primary',
                              layout=widgets.Layout(width='200px', height='36px'))
scenario_output = widgets.Output()

def on_generate(b):
    with scenario_output:
        clear_output(wait=True)
        scope = scope_selector.value
        if mode_selector.value == 'smarterdx':
            for _ in range(20):
                s = ipu.generate_scenario(scope_filter=scope)
                if s['archetype_type'] == 'Healthcare AI':
                    break
        else:
            s = ipu.generate_scenario(scope_filter=scope)
        _state['scenario'] = s
        _state['phase_results'] = {}
        display(HTML(ui.format_scenario_html(s)))

generate_btn.on_click(on_generate)
display(widgets.VBox([scope_selector, mode_selector, generate_btn, scenario_output]))

In [4]:
# ── Five-Phase Walkthrough ──
display(HTML('''<div class="ipu-section"><h2>Five-Phase Walkthrough</h2>
<p>Write or record your response at each phase. Submit to get scored, or use Hint / Example if stuck.</p></div>'''))

for phase in ipu.PHASES:
    w = ui.make_phase_widget(phase, _state, SESSION_ID, ipu)
    display(w)

In [ ]:
# ── Session Summary ──
display(HTML('<div class="ipu-section"><h2>Session Summary</h2><p>Run this cell after completing phases to see overall performance.</p></div>'))

phase_scores = sorted(_state.get('phase_results', {}).values(), key=lambda x: x['phase_num'])

if phase_scores:
    display(HTML(ui.format_session_summary_html(phase_scores)))

    # Radar chart
    import matplotlib.pyplot as plt
    import numpy as np

    all_dims = {}
    for s in phase_scores:
        for dim, val in s.get('scores', {}).items():
            clean = dim.replace('_', ' ').title()
            all_dims.setdefault(clean, []).append(val)

    if all_dims:
        names = list(all_dims.keys())
        avgs = [round(sum(v)/len(v), 1) for v in all_dims.values()]
        angles = np.linspace(0, 2*np.pi, len(names), endpoint=False).tolist()
        avgs_plot = avgs + [avgs[0]]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
        ax.fill(angles, avgs_plot, alpha=0.2, color='#2E86AB')
        ax.plot(angles, avgs_plot, 'o-', linewidth=2, color='#2E86AB')
        ax.set_ylim(0, 5)
        ax.set_yticks([1, 2, 3, 4, 5])
        ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=9)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(names, fontsize=9, wrap=True)
        overall = round(sum(s['composite'] for s in phase_scores) / len(phase_scores), 1)
        ax.set_title(f'Skill Profile — Overall: {overall}/5', fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_DIR, f'{SESSION_ID}_radar.png'), dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("No scored phases yet. Complete the walkthrough above first.")

In [ ]:
# ── Save Session ──
saved_path = ipu.save_session_log()
if saved_path:
    display(HTML(f'''<div style="background:#dcfce7; border-left:4px solid #16a34a; padding:10px; border-radius:6px; font-size:13px;">
        Session saved: <code>{saved_path}</code></div>'''))

---

**To practice again:** re-run the scenario generator cell above, then work through the phases.

| Phase | Focus | Key Deliverables |
|-------|-------|-----------------|
| 1. Discovery | Size the opportunity | Market sizing, segmentation, competitive landscape |
| 2. Validation | Define success | North Star, metrics tree, baselines, kill criteria |
| 3. Build | Instrument | Event taxonomy, dashboards, data quality |
| 4. Rollout | Measure impact | Experiments, adoption funnels, guardrails |
| 5. Scale | Optimize | Retention, product-market fit, growth targeting |